# GPT Action 库：Jira

## 简介

本页面为开发者针对特定应用构建 GPT Action 提供说明和指南。在继续之前，请先熟悉以下内容： 
- [GPT Actions 简介](https://platform.openai.com/docs/actions)
- [GPT Actions 库简介](https://platform.openai.com/docs/actions/actions-library)
- [从零构建 GPT Action 的示例](https://platform.openai.com/docs/actions/getting-started)

这个 GPT Action 概述了如何连接 Jira（Atlassian 的项目与工单管理工具）。该 Action 以用户身份运行，允许用户读写指定项目中的 issue。

### 价值与业务用例示例

**价值**：用户现在可以利用 ChatGPT 的自然语言能力直接连接 Jira Cloud

**用例示例**： 
- 用户可以加载某个项目的近期 issue，并使用 ChatGPT 提供解决方案
- 用户可以通过指示 ChatGPT 来创建和修改 issue 及子任务，并将其分配给特定用户

## 应用信息

### 应用关键链接

开始之前，请先查看该应用的以下链接：
- 应用网站：https://<YOUR_SUBDOMAIN>.atlassian.net/jira
- 应用 API 文档：https://developer.atlassian.com/cloud/jira/platform/rest/v3/intro/
- 应用 OAuth 2.0 文档：https://developer.atlassian.com/cloud/jira/platform/oauth-2-3lo-apps/

### 应用前提条件

开始之前，请确保在你的应用环境中完成以下步骤：
- 确保你拥有在 [Atlassian Cloud Developer Console](https://developer.atlassian.com/console/myapps/) 中创建应用的访问权限和许可

## ChatGPT 步骤

### 自定义 GPT 指令 

创建自定义 GPT 后，将下面的文本复制到 Instructions 面板中。有疑问？请查看[入门示例](https://platform.openai.com/docs/actions/getting-started)，了解这一步的详细操作。

In [ ]:
**Context**: you are specialized GPT designed to create and edit issues through API connections to Jira Cloud. This GPT can create, read, and edit project issues based on user instructions.

**Instructions**:
- When asked to perform a task, use the available actions via the api.atlassian.com API.
- When asked to create an issue, use the user's input to synthesize a summary and description and file the issue in JIRA.
- When asked to create a subtask, assume the project key and parent issue key of the currently discussed issue. Clarify with if this context is not available.
- When asked to assign an issue or task to the user, first use jql to query the current user's profile and use this account as the assignee. 
- Ask for clarification when needed to ensure accuracy and completeness in fulfilling user requests.

### OpenAPI Schema 

创建自定义 GPT 后，将下面的文本复制到 Actions 面板中。有疑问？请查看[入门示例](https://platform.openai.com/docs/actions/getting-started)，了解这一步的详细操作。

*注意：将 url 中的占位符 <CLOUD_ID> 替换为你的云环境的唯一 ID。你可以访问 https://<YOUR_SUBDOMAIN>.atlassian.net/_edge/tenant_info 找到该值*


In [ ]:
openapi: 3.1.0
info:
  title: Jira API
  description: API for interacting with Jira issues and sub-tasks.
  version: 1.0.0
servers:
  - url: https://api.atlassian.com/ex/jira/<CLOUD_ID>/rest/api/3
    description: Jira Cloud API
components:
  securitySchemes:
    OAuth2:
      type: oauth2
      flows:
        authorizationCode:
          authorizationUrl: https://auth.atlassian.com/authorize
          tokenUrl: https://auth.atlassian.com/oauth/token
          scopes:
            read:jira-user: Read Jira user information
            read:jira-work: Read Jira work data
            write:jira-work: Write Jira work data
  schemas:
    Issue:
      type: object
      properties:
        id:
          type: string
        key:
          type: string
        fields:
          type: object
          properties:
            summary:
              type: string
            description:
              type: string
            issuetype:
              type: object
              properties:
                name:
                  type: string
paths:
  /search:
    get:
      operationId: getIssues
      summary: Retrieve a list of issues
      parameters:
        - name: jql
          in: query
          required: false
          schema:
            type: string
        - name: startAt
          in: query
          required: false
          schema:
            type: integer
        - name: maxResults
          in: query
          required: false
          schema:
            type: integer
      responses:
        '200':
          description: A list of issues
          content:
            application/json:
              schema:
                type: object
                properties:
                  issues:
                    type: array
                    items:
                      $ref: '#/components/schemas/Issue'
  /issue:
    post:
      operationId: createIssue
      summary: Create a new issue
      requestBody:
        required: true
        content:
          application/json:
            schema:
              type: object
              properties:
                fields:
                  type: object
                  properties:
                    project:
                      type: object
                      properties:
                        key:
                          type: string
                    summary:
                      type: string
                    description:
                      type: string
                    issuetype:
                      type: object
                      properties:
                        name:
                          type: string
      responses:
        '201':
          description: Issue created successfully
          content:
            application/json:
              schema:
                $ref: '#/components/schemas/Issue'
  /issue/{issueIdOrKey}:
    get:
      operationId: getIssue
      summary: Retrieve a specific issue
      parameters:
        - name: issueIdOrKey
          in: path
          required: true
          schema:
            type: string
      responses:
        '200':
          description: Issue details
          content:
            application/json:
              schema:
                $ref: '#/components/schemas/Issue'
    put:
      operationId: updateIssue
      summary: Update an existing issue
      parameters:
        - name: issueIdOrKey
          in: path
          required: true
          schema:
            type: string
      requestBody:
        required: true
        content:
          application/json:
            schema:
              type: object
              properties:
                fields:
                  type: object
                  properties:
                    summary:
                      type: string
                    description:
                      type: string
                    issuetype:
                      type: object
                      properties:
                        name:
                          type: string
      responses:
        '204':
          description: Issue updated successfully
  /issue:
    post:
      operationId: createSubTask
      summary: Create a sub-task for an issue
      requestBody:
        required: true
        content:
          application/json:
            schema:
              type: object
              properties:
                fields:
                  type: object
                  properties:
                    project:
                      type: object
                      properties:
                        key:
                          type: string
                    parent:
                      type: object
                      properties:
                        key:
                          type: string
                    summary:
                      type: string
                    description:
                      type: string
                    issuetype:
                      type: object
                      properties:
                        name:
                          type: string
      responses:
        '201':
          description: Sub-task created successfully
          content:
            application/json:
              schema:
                $ref: '#/components/schemas/Issue'
security:
  - OAuth2:
      - read:jira-user
      - read:jira-work
      - write:jira-work


## 身份验证说明

以下是设置 Jira 身份验证的说明。有疑问？请查看[入门示例](https://platform.openai.com/docs/actions/getting-started)，了解这一步的详细操作。


### Jira 步骤

1. <b>创建应用</b>：第一步是在 Jira 中创建一个用于与 ChatGPT 集成的新应用。方法是访问 [Atlassian Developer Console](https://developer.atlassian.com/console/myapps/)，点击 **Create** 并选择 **OAuth 2.0 Integration**。 

![gptactions_jira_devconsole.png](../../../images/gptactions_jira_devconsole.png)

然后输入集成的名称并点击 **Create**。

![gptactions_jira_newapplication.png](../../../images/gptactions_jira_newapplication.png)

2. <b>定义权限</b>：接下来需要为应用提供所需的权限。在新应用中，从侧边栏打开 **Permissions** 菜单，找到 **Jira API**，点击 **Add**，然后点击 **Configure**。

![gptactions_jira_permissions.png](../../../images/gptactions_jira_permissions.png)

所需权限取决于 GPT 的预期功能。在本场景中，我们希望读写 Jira issue，因此点击 **Edit Scopes**，在 **Jira platform REST API** 下选择以下 scope：

- read:jira-work
- write:jira-work
- read:jira-user

选择完成后，点击 **Save**

![gptactions_jira_scopes.png](../../../images/gptactions_jira_scopes.png)

3. <b>配置占位回调 URL</b>：为了完成下一步并获取用于在 ChatGPT 与 Jira 之间进行安全身份验证的 **Client ID** 和 **Secret**，我们需要先添加一个占位回调 URL。方法是点击侧边栏中的 **Authorization**，然后点击 **OAuth 2.0 (3LO)** 旁边的 **Configure**。在这里输入一个占位 URL 并点击 **Save Changes**。


![gptactions_jira_placeholder.png](../../../images/gptactions_jira_placeholder.png)

4. <b>应用 Client ID/Secret</b>：下一步是找到用于在 ChatGPT 与 Jira 之间进行安全身份验证的 **Client ID** 和 **Secret**。点击侧边栏中的 **Settings** 并向下滚动到 **Authentication Details**，即可找到这些值。 

      请保持此页面打开，因为下一阶段的配置需要用到这些值！

![gptactions_jira_clientsecret.png](../../../images/gptactions_jira_clientsecret.png)

### 在 ChatGPT 中

在 ChatGPT 中，点击 "Authentication" 并选择 **"OAuth"**。填入以下信息。 

- **Client ID**：Jira 配置**第 3 步**中的 **Client ID**
- **Client Secret**：Jira 配置**第 3 步**中的 **Secret**
- **Authorization URL**：https://auth.atlassian.com/authorize
- **Token URL**：https://auth.atlassian.com/oauth/token
- **Scope**：read:jira-work write:jira-work read:jira-user 
- **Token Exchange Method**：Default (POST Request)

### Action 后续步骤

在 ChatGPT 中设置好身份验证后，请在应用中按照以下步骤完成 Action 的最终配置。 

- 从 GPT Action 中复制回调 URL

![gptactions_jira_redirect.png](../../../images/gptactions_jira_redirect.png)

- 在 Atlassian Developer Console 中你的应用里，进入侧边栏的 **Authorization** 标签页，点击 **OAuth 2.0 (3L0)** 旁边的 **Configure**，并在 **Callback URL** 下添加你的回调 URL


![gptactions_jira_callback.png](../../../images/gptactions_jira_callback.png)

### 常见问题与故障排查

- **回调 URL 错误**：如果在 ChatGPT 中遇到回调 URL 错误，请仔细检查 Callback URL 的值，因为它偶尔会因身份验证设置的更改而变化


*有希望我们优先支持的集成吗？我们的集成中有错误吗？请在我们的 github 上提交 PR 或 issue，我们会查看。*
